# F0 Comparison: Target RMVPE vs Prior RMVPE vs Prior USTX Pitch Curves

Compare three F0 distributions on validation examples:
1. **Target RMVPE** — F0 extracted from the ground-truth target audio (what the model trains on)
2. **Prior RMVPE** — F0 extracted from the OpenUtau prior WAV
3. **Prior USTX** — F0 synthesized from USTX pitch control points (new method)

In [ ]:
# ── Configuration ──

MANIFEST_PATH = "../Data/Rachie/manifest.csv"
DATA_DIR = "../Data/Rachie"
PHONESET_PATH = "../SoulX-Singer/soulxsinger/utils/phoneme/phone_set.json"
RMVPE_MODEL_PATH = "../SoulX-Singer/pretrained_models/SoulX-Singer-Preprocess/rmvpe/rmvpe.pt"

DEVICE = "auto"
N_SAMPLES = 6

# Validation split (must match training config)
MAX_DTW_COST = 100.0
VAL_FRACTION = 0.05
SEED = 42

# DTW cost range for sample selection
SAMPLE_DTW_MIN = 20.0
SAMPLE_DTW_MAX = 60.0

SR = 24000
HOP = 480
FRAME_SEC = HOP / SR

## Imports and Setup

In [ ]:
import sys, os
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
from IPython.display import Audio, display, HTML

# ── Path setup (mirrors VocaloFlow/inference/pipeline.py) ──
REPO_ROOT = str(Path("..").resolve())
VOCALOFLOW_DIR = os.path.join(REPO_ROOT, "VocaloFlow")
SOULX_DIR = os.path.join(REPO_ROOT, "SoulX-Singer")

for p in [REPO_ROOT, VOCALOFLOW_DIR, SOULX_DIR]:
    if p not in sys.path:
        sys.path.insert(0, p)

from utils.data_helpers import load_manifest, filter_manifest, split_by_song
from utils.resample import resolve_phoneme_indirection, resample_1d

from inference.pipeline import (
    parse_ustx, extract_f0_rmvpe,
    synthesize_f0_from_ustx_pitch, synthesize_f0_from_midi,
    build_phoneme_ids, get_note_coverage_voicing,
    get_voiced_mask, _interpolate_pitch_points,
)

if DEVICE == "auto":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
else:
    device = torch.device(DEVICE)
print(f"Using device: {device}")

## Load Manifest and Select Validation Samples

In [ ]:
manifest = load_manifest(MANIFEST_PATH, DATA_DIR)
filtered = filter_manifest(manifest, max_dtw_cost=MAX_DTW_COST)
train_df, val_df = split_by_song(filtered, val_fraction=VAL_FRACTION, seed=SEED)

candidates = val_df[
    (val_df["dtw_cost"] >= SAMPLE_DTW_MIN) &
    (val_df["dtw_cost"] <= SAMPLE_DTW_MAX)
].copy()

def _has_pipeline_inputs(row):
    d = os.path.dirname(row["prior_mel_path"])
    return all(os.path.exists(os.path.join(d, f))
               for f in ("prior.ustx", "prior.wav", "target.wav"))

candidates = candidates[candidates.apply(_has_pipeline_inputs, axis=1)]
samples = candidates.sample(n=min(N_SAMPLES, len(candidates)), random_state=SEED)

print(f"Validation set: {len(val_df)} chunks")
print(f"Candidates (DTW {SAMPLE_DTW_MIN}–{SAMPLE_DTW_MAX}, pipeline inputs present): {len(candidates)}")
print(f"\nSelected {len(samples)} samples:")
for _, row in samples.iterrows():
    print(f"  {row['dali_id'][:12]}…/{row['chunk_name']}  DTW={row['dtw_cost']:.1f}")

## Extract All Three F0 Curves Per Sample

In [ ]:
def _match_1d(x, n):
    return x[:n] if len(x) >= n else np.pad(x, (0, n - len(x)))

def mask_unvoiced(f0):
    out = f0.astype(np.float64).copy()
    out[out <= 0] = np.nan
    return out

all_f0_data = []

for idx, (_, row) in enumerate(samples.iterrows()):
    chunk_dir = os.path.dirname(row["prior_mel_path"])
    sample_id = f"{row['dali_id'][:12]}…/{row['chunk_name']}"
    print(f"\n{'=' * 60}")
    print(f"Sample {idx + 1}/{len(samples)}: {sample_id}")
    print(f"{'=' * 60}")

    # Target length from training F0
    f0_target_raw = np.load(row["f0_path"]).astype(np.float32)
    T = len(f0_target_raw)

    # 1. Target RMVPE (pre-computed training F0 from target audio)
    f0_target = _match_1d(f0_target_raw, T)

    # 2. Prior RMVPE (extract from prior WAV)
    prior_wav = os.path.join(chunk_dir, "prior.wav")
    f0_prior_rmvpe_raw = extract_f0_rmvpe(prior_wav, os.path.abspath(RMVPE_MODEL_PATH), str(device))
    f0_prior_rmvpe = resample_1d(f0_prior_rmvpe_raw, T, mode="linear").numpy().astype(np.float32)

    # 3. Prior USTX pitch curves (raw, no voicing zeroing)
    ustx_path = os.path.join(chunk_dir, "prior.ustx")
    notes_data = parse_ustx(ustx_path)
    f0_ustx = synthesize_f0_from_ustx_pitch(notes_data, T, voicing=None)
    f0_ustx = resample_1d(f0_ustx, T, mode="linear").numpy().astype(np.float32)

    target_wav = os.path.join(chunk_dir, "target.wav")

    all_f0_data.append({
        "sample_id": sample_id,
        "chunk_dir": chunk_dir,
        "T": T,
        "f0_target": f0_target,
        "f0_prior_rmvpe": f0_prior_rmvpe,
        "f0_ustx": f0_ustx,
        "notes_data": notes_data,
        "target_wav": target_wav,
        "prior_wav": prior_wav,
    })

    print(f"  T={T} frames ({T * FRAME_SEC:.2f}s)")
    for name, f0 in [("Target RMVPE", f0_target), ("Prior RMVPE", f0_prior_rmvpe), ("USTX", f0_ustx)]:
        voiced = f0 > 0
        v_mean = f0[voiced].mean() if voiced.any() else 0
        print(f"  {name:15s}: voiced={voiced.mean():.1%}, mean={v_mean:.0f}Hz")

print(f"\nExtracted F0 for {len(all_f0_data)} samples.")

## Per-Sample F0 Overlay + Audio

For each validation sample: overlay all three F0 curves, plus audio playback of target and prior.

In [ ]:
for data in all_f0_data:
    T = data["T"]
    t = np.arange(T) * FRAME_SEC

    display(HTML(f"<h3>{data['sample_id']}</h3>"))

    fig, ax = plt.subplots(figsize=(18, 5))
    ax.plot(t, mask_unvoiced(data["f0_target"]), color="tab:green", alpha=0.8,
            linewidth=1.2, label="Target RMVPE (ground truth)")
    ax.plot(t, mask_unvoiced(data["f0_prior_rmvpe"]), color="tab:blue", alpha=0.8,
            linewidth=1.2, label="Prior RMVPE (prior WAV)")
    ax.plot(t, mask_unvoiced(data["f0_ustx"]), color="tab:red", alpha=0.8,
            linewidth=1.2, label="USTX pitch curves (raw)")

    ax.set_xlabel("Time (s)")
    ax.set_ylabel("F0 (Hz)")
    ax.set_title(f"F0 Comparison — {data['sample_id']}")
    ax.legend(loc="upper right")
    ax.set_xlim(t[0], t[-1])
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    print("Target audio:")
    display(Audio(data["target_wav"]))
    print("Prior audio:")
    display(Audio(data["prior_wav"]))

## Per-Sample F0 — Separate Subplots

Each method on its own axis with shared y-scale for direct visual comparison.

In [ ]:
from matplotlib.patches import Rectangle

def midi_to_hz(midi):
    return 440.0 * (2.0 ** ((midi - 69) / 12.0))

NOTE_NAMES = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]

def midi_to_name(midi):
    return f"{NOTE_NAMES[midi % 12]}{midi // 12 - 1}"

for data in all_f0_data:
    T = data["T"]
    t = np.arange(T) * FRAME_SEC
    notes_data = data["notes_data"]
    ms_per_tick = notes_data["ms_per_tick"]

    curves = [
        ("Target RMVPE (ground truth)", data["f0_target"], "tab:green"),
        ("Prior RMVPE (prior WAV)",     data["f0_prior_rmvpe"], "tab:blue"),
        ("USTX pitch curves (raw)",     data["f0_ustx"], "tab:red"),
    ]

    all_voiced = np.concatenate([f[f > 0] for _, f, _ in curves])
    ylo = max(all_voiced.min() - 30, 0) if len(all_voiced) else 0
    yhi = all_voiced.max() + 30 if len(all_voiced) else 500

    fig, axes = plt.subplots(3, 1, figsize=(18, 10), sharex=True, sharey=True)
    fig.suptitle(f"F0 with MIDI Notes — {data['sample_id']}", fontsize=13)

    for ax, (label, f0, color) in zip(axes, curves):
        # Draw MIDI note rectangles + lyrics
        for note in notes_data["notes"]:
            start_s = note["position_ticks"] * ms_per_tick / 1000.0
            dur_s = note["duration_ticks"] * ms_per_tick / 1000.0
            note_hz = midi_to_hz(note["tone"])
            lyric = note["lyric"]

            rect = Rectangle(
                (start_s, note_hz - 3), dur_s, 6,
                linewidth=0.8, edgecolor="gray", facecolor="gold", alpha=0.25,
            )
            ax.add_patch(rect)

            ax.text(
                start_s + dur_s / 2, note_hz + 5,
                f"{lyric}\n{midi_to_name(note['tone'])}",
                ha="center", va="bottom", fontsize=6, color="dimgray",
            )

        ax.plot(t, mask_unvoiced(f0), color=color, linewidth=1.2)
        voiced = f0 > 0
        v_mean = f0[voiced].mean() if voiced.any() else 0
        ax.set_ylabel("F0 (Hz)")
        ax.set_title(f"{label}  —  voiced: {voiced.mean():.1%}, mean: {v_mean:.0f} Hz",
                      fontsize=10)
        ax.grid(True, alpha=0.3)

    axes[0].set_ylim(ylo, yhi)
    axes[-1].set_xlabel("Time (s)")
    plt.tight_layout()
    plt.show()

## Cents Difference: USTX vs Target, Prior RMVPE vs Target

For frames where both curves are voiced, show how far each method deviates from the ground-truth target F0.

In [ ]:
for data in all_f0_data:
    T = data["T"]
    t = np.arange(T) * FRAME_SEC
    ft, fp, fu = data["f0_target"], data["f0_prior_rmvpe"], data["f0_ustx"]

    # USTX vs Target
    both_ut = (ft > 0) & (fu > 0)
    cents_ustx = np.full(T, np.nan)
    cents_ustx[both_ut] = 1200 * np.log2(fu[both_ut] / ft[both_ut])

    # Prior RMVPE vs Target
    both_pt = (ft > 0) & (fp > 0)
    cents_prior = np.full(T, np.nan)
    cents_prior[both_pt] = 1200 * np.log2(fp[both_pt] / ft[both_pt])

    fig, axes = plt.subplots(2, 1, figsize=(18, 6), sharex=True)
    fig.suptitle(f"Cents Deviation from Target — {data['sample_id']}", fontsize=12)

    axes[0].plot(t, cents_prior, color="tab:blue", linewidth=0.8, alpha=0.8)
    axes[0].axhline(0, color="k", linewidth=0.5)
    axes[0].axhline(100, color="r", linestyle="--", alpha=0.3)
    axes[0].axhline(-100, color="r", linestyle="--", alpha=0.3)
    v = cents_prior[~np.isnan(cents_prior)]
    axes[0].set_ylabel("Cents")
    axes[0].set_title(f"Prior RMVPE vs Target — mean={np.mean(v):.1f}, std={np.std(v):.1f}, "
                       f"|>100|={np.mean(np.abs(v)>100)*100:.1f}%")
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(t, cents_ustx, color="tab:red", linewidth=0.8, alpha=0.8)
    axes[1].axhline(0, color="k", linewidth=0.5)
    axes[1].axhline(100, color="r", linestyle="--", alpha=0.3)
    axes[1].axhline(-100, color="r", linestyle="--", alpha=0.3)
    v = cents_ustx[~np.isnan(cents_ustx)]
    axes[1].set_ylabel("Cents")
    axes[1].set_xlabel("Time (s)")
    axes[1].set_title(f"USTX vs Target — mean={np.mean(v):.1f}, std={np.std(v):.1f}, "
                       f"|>100|={np.mean(np.abs(v)>100)*100:.1f}%")
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

## Aggregate Distribution Comparison

Pool all voiced F0 values across samples and compare distributions.

In [ ]:
# Pool all voiced F0 values
all_target = np.concatenate([d["f0_target"][d["f0_target"] > 0] for d in all_f0_data])
all_prior = np.concatenate([d["f0_prior_rmvpe"][d["f0_prior_rmvpe"] > 0] for d in all_f0_data])
all_ustx = np.concatenate([d["f0_ustx"][d["f0_ustx"] > 0] for d in all_f0_data])

fig, axes = plt.subplots(1, 2, figsize=(18, 5))

# Hz histogram
bins_hz = np.linspace(50, 800, 120)
axes[0].hist(all_target, bins=bins_hz, alpha=0.5, color="tab:green", label="Target RMVPE", density=True)
axes[0].hist(all_prior, bins=bins_hz, alpha=0.5, color="tab:blue", label="Prior RMVPE", density=True)
axes[0].hist(all_ustx, bins=bins_hz, alpha=0.5, color="tab:red", label="USTX", density=True)
axes[0].set_xlabel("F0 (Hz)")
axes[0].set_ylabel("Density")
axes[0].set_title("F0 Distribution (Hz) — All Samples Pooled")
axes[0].legend()

# MIDI note histogram (more musically meaningful)
def hz_to_midi(hz):
    return 69 + 12 * np.log2(hz / 440.0)

bins_midi = np.linspace(40, 90, 100)
axes[1].hist(hz_to_midi(all_target), bins=bins_midi, alpha=0.5, color="tab:green",
             label="Target RMVPE", density=True)
axes[1].hist(hz_to_midi(all_prior), bins=bins_midi, alpha=0.5, color="tab:blue",
             label="Prior RMVPE", density=True)
axes[1].hist(hz_to_midi(all_ustx), bins=bins_midi, alpha=0.5, color="tab:red",
             label="USTX", density=True)
axes[1].set_xlabel("MIDI note")
axes[1].set_ylabel("Density")
axes[1].set_title("F0 Distribution (MIDI note) — All Samples Pooled")
axes[1].legend()

plt.tight_layout()
plt.show()

# Aggregate cents deviation from target
all_cents_prior = []
all_cents_ustx = []
for d in all_f0_data:
    ft, fp, fu = d["f0_target"], d["f0_prior_rmvpe"], d["f0_ustx"]
    m_p = (ft > 0) & (fp > 0)
    m_u = (ft > 0) & (fu > 0)
    if m_p.any():
        all_cents_prior.append(1200 * np.log2(fp[m_p] / ft[m_p]))
    if m_u.any():
        all_cents_ustx.append(1200 * np.log2(fu[m_u] / ft[m_u]))

all_cents_prior = np.concatenate(all_cents_prior)
all_cents_ustx = np.concatenate(all_cents_ustx)

fig, ax = plt.subplots(figsize=(12, 5))
bins_c = np.linspace(-300, 300, 120)
ax.hist(all_cents_prior, bins=bins_c, alpha=0.6, color="tab:blue", label="Prior RMVPE vs Target", density=True)
ax.hist(all_cents_ustx, bins=bins_c, alpha=0.6, color="tab:red", label="USTX vs Target", density=True)
ax.axvline(0, color="k", linewidth=0.8)
ax.axvline(100, color="gray", linestyle="--", alpha=0.5)
ax.axvline(-100, color="gray", linestyle="--", alpha=0.5)
ax.set_xlabel("Cents deviation from Target")
ax.set_ylabel("Density")
ax.set_title("Aggregate Cents Deviation from Target F0")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Prior RMVPE vs Target:  mean={np.mean(all_cents_prior):+.1f}  std={np.std(all_cents_prior):.1f}  "
      f"|>100|={np.mean(np.abs(all_cents_prior)>100)*100:.1f}%")
print(f"USTX vs Target:         mean={np.mean(all_cents_ustx):+.1f}  std={np.std(all_cents_ustx):.1f}  "
      f"|>100|={np.mean(np.abs(all_cents_ustx)>100)*100:.1f}%")

## Voicing Agreement

In [ ]:
for data in all_f0_data:
    T = data["T"]
    t = np.arange(T) * FRAME_SEC
    ft, fp, fu = data["f0_target"], data["f0_prior_rmvpe"], data["f0_ustx"]

    v_target = (ft > 0).astype(float)
    v_prior = (fp > 0).astype(float)
    v_ustx = (fu > 0).astype(float)

    fig, axes = plt.subplots(4, 1, figsize=(18, 7), sharex=True)
    fig.suptitle(f"Voicing — {data['sample_id']}", fontsize=12)

    axes[0].fill_between(t, 0, v_target, alpha=0.6, color="tab:green", step="mid")
    axes[0].set_ylabel("Voiced")
    axes[0].set_title("Target RMVPE")
    axes[0].set_ylim(-0.1, 1.1)

    axes[1].fill_between(t, 0, v_prior, alpha=0.6, color="tab:blue", step="mid")
    axes[1].set_ylabel("Voiced")
    axes[1].set_title("Prior RMVPE")
    axes[1].set_ylim(-0.1, 1.1)

    axes[2].fill_between(t, 0, v_ustx, alpha=0.6, color="tab:red", step="mid")
    axes[2].set_ylabel("Voiced")
    axes[2].set_title("USTX (note coverage)")
    axes[2].set_ylim(-0.1, 1.1)

    disagree_prior = (v_target != v_prior).astype(float)
    disagree_ustx = (v_target != v_ustx).astype(float)
    axes[3].fill_between(t, 0, disagree_prior, alpha=0.5, color="tab:blue", step="mid",
                          label=f"Prior RMVPE ({disagree_prior.mean()*100:.1f}%)")
    axes[3].fill_between(t, 0, -disagree_ustx, alpha=0.5, color="tab:red", step="mid",
                          label=f"USTX ({disagree_ustx.mean()*100:.1f}%)")
    axes[3].set_ylabel("Disagree")
    axes[3].set_xlabel("Time (s)")
    axes[3].set_title("Disagreement with Target")
    axes[3].set_ylim(-1.1, 1.1)
    axes[3].legend(fontsize=8)

    plt.tight_layout()
    plt.show()

## Summary Statistics

In [ ]:
print("=" * 70)
print("F0 COMPARISON SUMMARY — Validation Samples")
print("=" * 70)

for data in all_f0_data:
    print(f"\n{data['sample_id']}  (T={data['T']})")
    ft, fp, fu = data["f0_target"], data["f0_prior_rmvpe"], data["f0_ustx"]
    for name, f0 in [("Target RMVPE", ft), ("Prior RMVPE", fp), ("USTX", fu)]:
        v = f0 > 0
        print(f"  {name:15s}: voiced={v.mean():.1%}  "
              f"range={f0[v].min():.0f}–{f0[v].max():.0f}Hz  "
              f"mean={f0[v].mean():.0f}Hz" if v.any() else f"  {name:15s}: no voiced frames")

    # Correlation with target
    corr_prior = np.corrcoef(ft, fp)[0, 1] if len(ft) > 1 else 0
    corr_ustx = np.corrcoef(ft, fu)[0, 1] if len(ft) > 1 else 0
    print(f"  Corr w/ target:  Prior RMVPE={corr_prior:.4f}  USTX={corr_ustx:.4f}")

print(f"\n{'=' * 70}")
print("AGGREGATE")
print(f"{'=' * 70}")
print(f"\nPrior RMVPE vs Target:  mean={np.mean(all_cents_prior):+.1f} cents  "
      f"std={np.std(all_cents_prior):.1f}  |>semitone|={np.mean(np.abs(all_cents_prior)>100)*100:.1f}%")
print(f"USTX vs Target:         mean={np.mean(all_cents_ustx):+.1f} cents  "
      f"std={np.std(all_cents_ustx):.1f}  |>semitone|={np.mean(np.abs(all_cents_ustx)>100)*100:.1f}%")